# papa-vision — end-to-end demo

A short walkthrough of the pipeline: load the potato dataset, inspect samples, build the from-scratch CNN, train briefly, evaluate, and visualise a Grad-CAM map.

Run inside the project environment: `uv run jupyter lab` (or `uv run jupyter notebook`). Make sure you have downloaded the data first with `make data`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import torch, numpy as np, matplotlib.pyplot as plt
from papavision.utils import get_device, set_seed
from papavision.data import get_dataloaders, IMAGENET_MEAN, IMAGENET_STD
from papavision.models import build_model, count_parameters
set_seed(0)
device = get_device()
print('device:', device)

## 1. Load the data and inspect class samples

In [ ]:
loaders, meta = get_dataloaders(img_size=128, batch_size=32)
print(meta)
from papavision import CLASS_LABELS
seen = {}
for img, label in loaders['test'].dataset:
    if label not in seen:
        arr = img.numpy().transpose(1, 2, 0)
        seen[label] = np.clip(arr * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)
    if len(seen) == 3:
        break
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for i in range(3):
    ax[i].imshow(seen[i]); ax[i].set_title(CLASS_LABELS[i]); ax[i].axis('off')
plt.show()

## 2. Build the from-scratch CNN

In [ ]:
model = build_model('custom_cnn', num_classes=3, width=32).to(device)
print('parameters:', count_parameters(model))
print(model)

## 3. Train a few epochs (quick demo — see `make train-all` for the full runs)

In [ ]:
from papavision.train import train
cfg = dict(model='custom_cnn', run_id='demo', seed=0, width=32, dropout=0.3,
           pretrained=False, freeze_backbone=False, img_size=128, batch_size=32,
           val_frac=0.15, test_frac=0.15, split_seed=42, aug_strength=1.0,
           epochs=5, lr=1e-3, weight_decay=1e-4, scheduler='cosine',
           label_smoothing=0.05, patience=4, device=str(device), deterministic=True)
record = train(cfg)
print('best val macro-F1:', record['best_val_macro_f1'])

## 4. Evaluate on the held-out test set

In [ ]:
from papavision.evaluate import collect_predictions, classification_metrics
out = collect_predictions(model, loaders['test'], device)
m = classification_metrics(out['y_true'], out['y_pred'], out['probs'])
print('accuracy: %.3f | macro-F1: %.3f | ECE: %.3f' % (m['accuracy'], m['macro_f1'], m['ece']))

## 5. Grad-CAM: where does the model look?

In [ ]:
from papavision.gradcam import GradCAM, denormalize
cam = GradCAM(model)
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
imgs = {}
for img, label in loaders['test'].dataset:
    if label not in imgs:
        imgs[label] = img
    if len(imgs) == 3:
        break
for i in range(3):
    heat = cam(imgs[i].unsqueeze(0).to(device), target_class=i)
    ax[i].imshow(denormalize(imgs[i], IMAGENET_MEAN, IMAGENET_STD))
    ax[i].imshow(heat, cmap='jet', alpha=0.45); ax[i].set_title(CLASS_LABELS[i]); ax[i].axis('off')
cam.remove(); plt.show()